# IMDb Review Sentiment Classification

## Business Context

Online reviews are a useful source of customer feedback, but reading thousands of them manually does not scale. A sentiment classifier can help a product, marketing, or content team quickly separate positive and negative feedback, track shifts in audience perception, and prioritize reviews that need closer attention.

## Objective

The goal of this notebook is to build a baseline neural-network classifier that predicts whether an IMDb movie review is positive or negative. The workflow covers label encoding, train/test splitting, text vectorization, model training, and evaluation on unseen reviews.

This is intentionally a compact baseline. The focus is on building a transparent end-to-end NLP pipeline and understanding what can be achieved with a simple multi-hot representation before moving to richer language models.

### Step 1: Import Required Libraries

Load the libraries used for data preparation, text vectorization, neural-network modeling, and evaluation.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0=all, 1=INFO, 2=WARNING, 3=ERROR
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
tf.random.set_seed(42)

### Step 2: Load the IMDb Review Dataset

Load the review dataset and inspect its structure before modeling. Each record contains the review text and its sentiment label. This first check is important because the downstream pipeline assumes one text feature and one binary target.

In [2]:
data=pd.read_excel('/data/IMDB_dataset.xlsx')
print(data.head())
print(data.shape)

                                              review sentiment
0  I thought this was a wonderful way to spend ti...  positive
1  Probably my all-time favorite movie, a story o...  positive
2  I sure would like to see a resurrection of a u...  positive
3  This show was an amazing, fresh & innovative i...  negative
4  Encouraged by the positive comments about this...  negative
(25000, 2)


### Step 3: Encode the Sentiment Labels

Convert the categorical sentiment labels into numeric values: negative reviews become `0` and positive reviews become `1`. This creates a target that can be used directly by the classification pipeline.

In [3]:
## .map() to replace all and store it to data['sentiment'].
## .fillna(0) for unmapped values.
## .astype(int) to define values as integer.
sentiment_map = {
    'positive': 1,
    'negative': 0
}
data['sentiment'] = data['sentiment'].map(sentiment_map).fillna(0).astype(int)
print(data['sentiment'].unique())

[1 0]


### Step 4: Define the Target Variable

Separate the sentiment column from the review text so the model inputs and prediction target can be handled independently.

In [4]:
y = data['sentiment']
print(y[0:10])

0    1
1    1
2    1
3    0
4    0
5    0
6    0
7    0
8    1
9    0
Name: sentiment, dtype: int64


In [5]:
print(data.shape)

(25000, 2)


### Step 5: Split the Data into Training and Test Sets

Create an 80/20 train-test split. The training set is used to learn the text representation and model parameters, while the test set is kept aside for an out-of-sample evaluation.

In [6]:
X_train,X_test,y_train,y_test = train_test_split(data['review'],y,test_size=0.2,random_state=42)
print(f"""
Train samples: {X_train.shape[0]}
Test samples: {X_test.shape[0]}
"""
)


Train samples: 20000
Test samples: 5000



### Step 6: Check the Sentiment Distribution

Review the class balance in the training set before fitting the model. The two classes are almost evenly represented, so accuracy is a reasonable first metric for this baseline and there is no immediate need to rebalance the data.

In [7]:
frequency=y_train.value_counts()/y_train.shape[0]
print(frequency)

sentiment
1    0.50135
0    0.49865
Name: count, dtype: float64


### Step 7: Convert the Targets to Dummy Vectors

Transform the binary labels into two-element one-hot vectors to match the two-unit softmax output layer used later in the neural network.

In [8]:
### Assign the output to y_train and y_test
y_train = tf.keras.utils.to_categorical(y_train,num_classes=2)
y_test = tf.keras.utils.to_categorical(y_test,num_classes=2)
print(y_train.shape)
print(y_test.shape)

(20000, 2)
(5000, 2)


### Step 8: Vectorize the Review Text

Use Keras `TextVectorization` with `output_mode="multi_hot"` and a vocabulary capped at 2,412 tokens. The vectorizer is adapted only on the training reviews so that the test set remains unseen during feature construction.

This representation is computationally efficient and works well as a baseline, although it treats text as a bag of tokens and therefore does not preserve word order or context.

In [9]:
max_tokens = 2412
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="multi_hot")

# Ensure X_train is a string Series for adaptation
# Re-splitting to guarantee a fresh string Series for adaptation
X_train_for_adapt, _, _, _ = train_test_split(data['review'], y, test_size=0.2, random_state=42)
text_vectorization.adapt(X_train_for_adapt)

In [10]:
print(y_train.shape)

(20000, 2)


In [11]:
X_train_vec = text_vectorization(X_train)
X_test_vec = text_vectorization(X_test)

print(X_train_vec.shape, X_test_vec.shape,y_train.shape, y_test.shape)

(20000, 2412) (5000, 2412) (20000, 2) (5000, 2)


### Step 9: Build the Neural Network

Create a compact feed-forward network for binary sentiment classification:

- An input layer matching the vectorized vocabulary size.
- A dense layer with 32 ReLU units to learn non-linear combinations of token features.
- A dropout layer with a rate of 0.5 to reduce overfitting.
- A two-unit softmax output layer for positive and negative sentiment probabilities.

The architecture is deliberately simple so the value of the text representation and baseline model can be evaluated without adding unnecessary complexity.

In [12]:
inputs = keras.Input(shape=(max_tokens, ))
x = keras.layers.Dense(32,activation = "relu")(inputs)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(2,activation = "softmax")(x)
model = keras.Model(inputs,outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 2412)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        77,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,282 (301.88 KB)

 Trainable params: 77,282 (301.88 KB)

 Non-trainable params: 0 (0.00 B)

### Step 10: Compile and Train the Model

Compile the network with the Adam optimizer and train it for five epochs. The stored training output provides a quick view of how accuracy and loss evolve as the model learns from the review vectors.

In [13]:
# Compile your model
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [14]:
# Fit your model
model.fit(x=X_train_vec, y=y_train,
          epochs=5,
          batch_size=32)


Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8162 - loss: 0.4039
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8850 - loss: 0.2813
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9046 - loss: 0.2440
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9141 - loss: 0.2190
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9226 - loss: 0.1965


### Step 11: Evaluate the Model

Evaluate the trained network on the held-out test set. The stored result is approximately **87% test accuracy**, which is a useful baseline for this binary sentiment task.

In [15]:
model.evaluate(x=X_test_vec, y=y_test)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8694 - loss: 0.3208


[0.32082194089889526, 0.8694000244140625]

## Conclusions

### Technical Takeaways

The baseline neural network reaches approximately **87% accuracy on unseen reviews**. For a relatively small feed-forward model, that is a solid result and confirms that even a simple multi-hot representation captures meaningful sentiment signals.

The main limitation is the text representation. Multi-hot encoding records whether tokens are present but discards word order, local context, and semantic relationships. Performance could be explored further with n-grams, embeddings, stronger regularization, hyperparameter tuning, or sequence-aware architectures. A production evaluation should also include precision, recall, F1 score, and an error analysis of false positives and false negatives rather than relying on accuracy alone.

### Business Takeaways

A model at this level can already support review triage, sentiment monitoring, and high-level trend analysis. Its value is strongest as a decision-support tool: it can reduce the amount of feedback that needs to be reviewed manually and help teams identify changes in audience perception more quickly.

Before operational use, the classification threshold and evaluation metrics should be aligned with the business cost of mistakes. For example, missing strongly negative feedback may be more costly than incorrectly flagging a neutral or positive review.

### Next Steps

The next iteration should focus on richer text representations and error analysis, then compare the incremental performance gain against the additional training and inference cost. The code in this notebook provides the full baseline pipeline and a practical starting point for those experiments.